In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql_1 = """
SELECT
    o.order_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    oi.is_free_gift,
    pa.campaign_id,
    p.product_m3_code,
    s.store_id,
    s.store_region,
    s.store_name,
    cr.review_id,
    cr.customer_id,
    cr.rating,
    cr.review_date,
    cr.product_id
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "PromotionActivity" pa
    ON o.campaign_id = pa.campaign_id
JOIN "ProductInfo" p
    ON oi.product_id = p.product_id
JOIN "StoreInfo" s
    ON o.store_id = s.store_id
JOIN "CustomerReview" cr
    ON o.order_id = cr.order_id
WHERE o.order_status IN ('Completed', 'Shipped');
"""

df_review_shipped = pd.read_sql(sql_1, engine)

for col in ["order_date", "review_date"]:
    if col in df_review_shipped.columns:
        df_review_shipped[col] = pd.to_datetime(
            df_review_shipped[col],
            errors="coerce"
        )

# 查看数据
df_review_shipped.to_parquet("9-1-review-overview.parquet",engine="fastparquet",index=False)
print("数据已保存到：9-1-review-overview.parquet")

数据已保存到：9-1-review-overview.parquet


### Complaint的数据输出

In [2]:
# 查询数据
sql_2 = """
SELECT
    o.order_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    oi.is_free_gift,
    pa.campaign_id,
    p.product_m3_code,
    s.store_id,
    s.store_region,
    s.store_name,
    cc.complaint_id,
    cc.customer_id,
    cc.complaint_date,
    cc.complaint_type,
    cc.complaint_severity,
    cc.resolution_status,
    cc.resolution_date,
    cc.product_id
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "PromotionActivity" pa
    ON o.campaign_id = pa.campaign_id
JOIN "ProductInfo" p
    ON oi.product_id = p.product_id
JOIN "StoreInfo" s
    ON o.store_id = s.store_id
JOIN "CustomerComplaint" cc
    ON o.order_id = cc.order_id
WHERE o.order_status IN ('Completed', 'Shipped');
"""

df_complaint_shipped = pd.read_sql(sql_2, engine)

# 查看数据

# 统一处理日期列
for col in ["order_date", "complaint_date", "resolution_date"]:
    if col in df_complaint_shipped.columns:
        df_complaint_shipped[col] = pd.to_datetime(
            df_complaint_shipped[col],
            errors="coerce"
        )

df_complaint_shipped.to_parquet(
    "9-1-complaint-overview.parquet",
    index=False,
    engine="pyarrow"
)
print("数据已保存到：9-1-complaint-overview.parquet")



数据已保存到：9-1-complaint-overview.parquet
